In [85]:
import pandas as pd
import numpy as np
import json
from scipy.stats import linregress
import matplotlib.pyplot as plt

# 1. Load the clean data with Kalman filter results
df = pd.read_csv('../data/strategy_results.csv', index_col=0, parse_dates=True)
# Map NB02 column names to what NB03/04 expect
df = df.rename(columns={'spread': 'spread_kalman', 'beta': 'beta_kalman'})

df = df.dropna().copy()

# ==========================================================
# VARIANCE ON THE SPREAD LEVEL
# ==========================================================
# The AS model needs σ² of the mid-price (spread) LEVEL, not of its increments.
# Using diff().var() gives ~1e-7 which kills the risk premium entirely.
# Using level var() gives ~1e-3 which is what the model actually needs.

df['variance'] = df['spread_kalman'].rolling(window=120).var()
df['variance'] = df['variance'].shift(1)     # Prevent look-ahead
df['variance'] = df['variance'].bfill()       # Fill initial NaN
df.loc[df['variance'] <= 0, 'variance'] = 1e-8

# Diagnostic: print the scale of variance
print(f"Variance stats (LEVEL, not diff):")
print(f"  Mean:   {df['variance'].mean():.6f}")
print(f"  Median: {df['variance'].median():.6f}")
print(f"  Std:    {df['variance'].std():.6f}")

# 2. Isolate the Train Set
df_train = df[df.index < '2024-01-01'].copy()
print(f"\nTrain Set loaded: {len(df_train)} rows from {df_train.index[0].date()} to {df_train.index[-1].date()}")

Variance stats (LEVEL, not diff):
  Mean:   0.065797
  Median: 0.027016
  Std:    0.166943

Train Set loaded: 402167 rows from 2020-01-02 to 2023-12-29


In [86]:
# ==============================================================================
# BROKER-AWARE MARKET MAKER (ALL FIXES APPLIED)
# ==============================================================================

class AsymptoticMarketMaker:
    def __init__(self, gamma, kappa, max_inventory=50):
        self.gamma = gamma          
        self.kappa = kappa          
        self.max_inventory = max_inventory
        self.q = 0                  
        
    def get_quotes(self, mid_price, variance, min_profitable_delta):
        # 1. Reservation Price (adjusts for inventory risk)
        r = mid_price - (self.q * self.gamma * variance)
        
        # 2. Optimal Mathematical Half-Spread
        liquidity_premium = (1 / self.gamma) * np.log(1 + (self.gamma / self.kappa))
        risk_premium = 0.5 * self.gamma * variance
        as_delta = liquidity_premium + risk_premium
        
        # 3. Fee Floor — never quote tighter than cost
        delta = max(as_delta, min_profitable_delta)
        
        # 4. Limit Orders
        bid = r - delta
        ask = r + delta
        
        # 5. Inventory Hard Stop
        if self.q >= self.max_inventory:
            bid = -np.inf
        elif self.q <= -self.max_inventory:
            ask = np.inf
            
        return bid, ask, r

def run_market_maker_sim(data_slice, gamma_val, kappa_val, max_inv=50):
    agent = AsymptoticMarketMaker(gamma=gamma_val, kappa=kappa_val, max_inventory=max_inv)
    cash = 0.0
    
    contract_multiplier = 100 
    alpaca_fee_per_share = 0.005 
    minimum_pure_profit = 0.005 
    
    spread_kalman = data_slice['spread_kalman'].values
    variance = data_slice['variance'].values
    beta_kalman = data_slice['beta_kalman'].values
    
    n = len(data_slice)
    equity_curve = np.zeros(n - 1)
    
    for i in range(n - 1):
        total_physical_shares = contract_multiplier + (contract_multiplier * beta_kalman[i])
        real_commission = total_physical_shares * alpaca_fee_per_share
        
        commission_per_spread_unit = real_commission / contract_multiplier
        min_profitable_delta = commission_per_spread_unit + minimum_pure_profit
        
        bid, ask, r = agent.get_quotes(spread_kalman[i], variance[i], min_profitable_delta)
        next_sync_price = spread_kalman[i+1]
        
        # ============================================================
        # elif PREVENTS DOUBLE-FILL IN SAME BAR
        # ============================================================
        if next_sync_price <= bid and agent.q < agent.max_inventory and bid != -np.inf:
            agent.q += 1
            cash -= (bid * contract_multiplier) + real_commission
            
        elif next_sync_price >= ask and agent.q > -agent.max_inventory and ask != np.inf:
            agent.q -= 1
            cash += (ask * contract_multiplier) - real_commission
            
        equity_curve[i] = cash + (agent.q * next_sync_price * contract_multiplier)
        
    equity_series = pd.Series(equity_curve, index=data_slice.index[1:])
    daily_pnl = equity_series.resample('B').last().diff().dropna()
    
    std_dev = daily_pnl.std()
    if std_dev > 0:
        return (daily_pnl.mean() / std_dev) * np.sqrt(252)
    return 0.0

In [87]:
# ==========================================================
# 1. EMPIRICAL KAPPA (κ) CALIBRATION — FIXED RANGE
# ==========================================================
print("Calibrating Market Microstructure Parameters (Empirical)...")

spread_diff = df_train['spread_kalman'].diff().abs().dropna()
spread_std = df_train['spread_kalman'].std()

# Match delta range to ACTUAL spread scale
# Instead of 0.01-0.15 (arbitrary), use range based on spread statistics
# This prevents fitting κ on the deep tail where probabilities collapse
# Delta range based on actual spread MOVEMENTS (not level)
# The spread level is ~$27 but minute-to-minute moves are much smaller
spread_move_std = spread_diff.std()
delta_min = max(0.1 * spread_move_std, 0.001)
delta_max = min(5.0 * spread_move_std, 3.0 * spread_std)
deltas = np.linspace(delta_min, delta_max, 30)

lambdas = []
for d in deltas:
    prob_fill = (spread_diff > d).mean()
    lambdas.append(prob_fill)

# Filter to valid (non-zero) fill probabilities
valid_idx = [i for i, l in enumerate(lambdas) if l > 0.001]  # require > 0.1% fill rate
valid_deltas = deltas[valid_idx]
valid_lambdas = np.array(lambdas)[valid_idx]

if len(valid_deltas) > 2:
    slope, intercept, r_value, _, _ = linregress(valid_deltas, np.log(valid_lambdas))
    kappa_empirical = -slope
    print(f"  κ regression R²: {r_value**2:.4f}")
else:
    # Fallback: use median-based estimate
    median_move = spread_diff.median()
    kappa_empirical = 1.0 / median_move if median_move > 0 else 10.0
    print(f"  κ estimated from median spread move (fallback)")

print(f"  Spread std: {spread_std:.6f}")
print(f"  Delta range: [{delta_min:.4f}, {delta_max:.4f}]")

# ==========================================================
# 2. GAMMA (γ) CALIBRATION — USING LEVEL VARIANCE
# ==========================================================
max_inventory = 50
avg_variance = df_train['variance'].mean()
avg_std_dev = np.sqrt(avg_variance)

# γ ensures inventory penalty at max position ≈ 1 std dev of spread
gamma_empirical = 1.0 / (max_inventory * avg_std_dev)

print(f"\n--- IN-SAMPLE CALIBRATION RESULTS ---")
print(f"True Market Liquidity (κ): {kappa_empirical:.4f}")
print(f"Optimal Risk Aversion (γ): {gamma_empirical:.4f}")
print(f"Avg Variance (LEVEL):      {avg_variance:.6f}")
print(f"Avg Std Dev:               {avg_std_dev:.6f}")

# Diagnostic: what does the AS model produce with these params?
test_liq_premium = (1 / gamma_empirical) * np.log(1 + (gamma_empirical / kappa_empirical))
test_risk_premium = 0.5 * gamma_empirical * avg_variance
print(f"\n--- AS MODEL DIAGNOSTICS ---")
print(f"Liquidity premium (1/γ · log(1+γ/κ)):  {test_liq_premium:.6f}")
print(f"Risk premium (0.5 · γ · σ²):            {test_risk_premium:.6f}")
print(f"Total half-spread (at avg variance):     {test_liq_premium + test_risk_premium:.6f}")

# Run the simulation
empirical_sharpe = run_market_maker_sim(df_train, gamma_val=gamma_empirical, kappa_val=kappa_empirical)
print(f"\nSharpe Ratio (Corrected): {empirical_sharpe:.3f}")

Calibrating Market Microstructure Parameters (Empirical)...
  κ regression R²: 0.9274
  Spread std: 8.667142
  Delta range: [0.0124, 0.6219]

--- IN-SAMPLE CALIBRATION RESULTS ---
True Market Liquidity (κ): 7.2843
Optimal Risk Aversion (γ): 0.0840
Avg Variance (LEVEL):      0.056655
Avg Std Dev:               0.238022

--- AS MODEL DIAGNOSTICS ---
Liquidity premium (1/γ · log(1+γ/κ)):  0.136496
Risk premium (0.5 · γ · σ²):            0.002380
Total half-spread (at avg variance):     0.138876

Sharpe Ratio (Corrected): -7.076


In [88]:
# ==========================================================
# OPTIONAL: GRID SEARCH AROUND EMPIRICAL VALUES
# ==========================================================
# If the empirical Sharpe is still poor, search around the calibrated values

print("Running grid search around empirical calibration...")

gamma_range = gamma_empirical * np.array([0.1, 0.25, 0.5, 1.0, 2.0, 4.0, 10.0])
kappa_range = kappa_empirical * np.array([0.1, 0.25, 0.5, 1.0, 2.0, 4.0])

best_sharpe = -np.inf
best_gamma = gamma_empirical
best_kappa = kappa_empirical

results_grid = []

for g in gamma_range:
    for k in kappa_range:
        sharpe = run_market_maker_sim(df_train, gamma_val=g, kappa_val=k)
        results_grid.append({'gamma': g, 'kappa': k, 'sharpe': sharpe})
        if sharpe > best_sharpe:
            best_sharpe = sharpe
            best_gamma = g
            best_kappa = k

print(f"\n--- GRID SEARCH RESULTS ---")
print(f"Best γ: {best_gamma:.6f}")
print(f"Best κ: {best_kappa:.4f}")
print(f"Best Sharpe: {best_sharpe:.3f}")
print(f"(Empirical baseline was: {empirical_sharpe:.3f})")

# Use the best parameters
final_gamma = best_gamma
final_kappa = best_kappa
final_sharpe = best_sharpe

Running grid search around empirical calibration...

--- GRID SEARCH RESULTS ---
Best γ: 0.084026
Best κ: 0.7284
Best Sharpe: 1.447
(Empirical baseline was: -7.076)


In [89]:
# ==========================================================
# SAVE OPTIMAL PARAMETERS
# ==========================================================
import os
os.makedirs('../results', exist_ok=True)

optimal_dict = {
    'gamma': final_gamma,
    'kappa': final_kappa,
    'train_sharpe': final_sharpe,
    'gamma_empirical': gamma_empirical,
    'kappa_empirical': kappa_empirical,
    'empirical_sharpe': empirical_sharpe,
    'avg_variance_level': avg_variance,
    'spread_std': spread_std,
}

with open('../results/optimal_params.json', 'w') as f:
    json.dump(optimal_dict, f, indent=4)

print(f"Parameters saved to ../results/optimal_params.json")
print(f"  γ = {final_gamma:.6f}")
print(f"  κ = {final_kappa:.4f}")
print(f"  Train Sharpe = {final_sharpe:.3f}")

Parameters saved to ../results/optimal_params.json
  γ = 0.084026
  κ = 0.7284
  Train Sharpe = 1.447
